# Lecture 3: Sampling in time, space and populations

**Guiding idea.** Lecture 2 was about sampling **molecules**: few UMIs per cell or bin, many zeros,
single genes that cannot be trusted. Here we move up one level and ask *what* we actually measured.
Every dataset is a sample taken at several nested levels:

```
time point → organoid / batch → section → cell or bin → molecules
```

Each level adds its own variance and its own bias. The question to keep asking is:
**at which level am I replicating, and at which level do I want to generalise?**

Same way of working as in Lectures 1 and 2: read the question, complete the **decomposition tree**,
write **one function per leaf** and a root function that calls them.

| Part | Exercises | Data |
|---|---|---|
| Time | 1 — the peak that wasn't there · 2 — how old is this organoid? · 3 — pseudotime from a snapshot | simulations |
| Space | 4 — a slice of an organoid · 5 — neighbouring bins are not independent | simulation + `Datasets/visiumhd_mouse_brain_cortex_2um.npz` |
| Populations | 6 — how many cells do I need? · 7 — population traps | simulations |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, ndimage, sparse
from scipy.sparse import csgraph
from scipy.spatial import cKDTree

rng = np.random.default_rng(2026)

## Part A — Sampling in time

Every time point is **destructive**: we never follow the same organoid over time, we compare different
organoids harvested at different ages. A time course is therefore a handful of snapshots, and what
happens *between* snapshots is invisible.

## Exercise 1 — The peak that wasn't there (simulation)

**Question:** a gene (think of *EOMES* in intermediate progenitors) has a transient peak of expression
around day 45 that lasts about two weeks. With a time course of 3, 5 or 10 time points between day 10 and
day 120, do we see the peak?

```
Can a time course see a transient peak?
├── The truth           → true_expression(t): baseline + Gaussian peak (centre 45 d, width 6 d)
├── The design          → timepoints(n, start=10, end=120): n equally spaced days
├── The measurement     → measure(days, n_organoids=3): true value + biological noise (CV 20%)
├── The summary         → observed peak height / true peak height
└── Plot                → true curve + measured points, one panel per design
```

**To do:** write the four functions and `peak_recovery(n_points, shift=0)`, which returns the
observed/true ratio for a design shifted by `shift` days. Then compute it for `n_points` in 3…15 and
for many random shifts of the schedule.

*Discussion question:* in a published time course of organoids (30, 60, 90, 120 days, a very common
design) could you claim that a gene is "never expressed"? What is the minimum spacing you need, given the
width of the peak? (hint: think of the sampling theorem.)

In [ ]:
def true_expression(t, baseline=1.0, amplitude=10.0, centre=45.0, width=6.0):
    # TODO
    pass

def timepoints(n, start=10.0, end=120.0, shift=0.0):
    # TODO
    pass

def measure(days, n_organoids=3, cv=0.2, rng=rng):
    # TODO: true value times log-normal noise, one column per organoid
    pass

def peak_recovery(n_points, shift=0.0):
    # TODO
    pass

def plot_designs(designs=(3, 5, 10)):
    # TODO
    pass

## Exercise 2 — How old is this organoid? (simulation)

**Question:** organoids develop more slowly than the fetal cortex, and not all at the same speed.
If we align "day 60 of an organoid" to a fixed post-conception week, how wrong can we be?
Can we estimate the biological age from the transcriptome instead (the time-wise benchmarking against
BrainSpan of Nico's Lecture 2)?

The simulated reference has 5 gene modules of 10 genes each, with different dynamics between post-conception
week (PCW) 8 and 24: progenitors (decreasing), intermediate progenitors (early peak), deep-layer neurons
(middle peak), upper-layer neurons (late rise), glia (late rise).

```
Biological age of an organoid
├── Reference         → reference_profile(pcw): 50 genes, smooth in PCW
├── Organoids         → simulate_organoid(day, speed): biological PCW = 6 + speed * day / 7, plus noise
├── Naive alignment   → calendar_pcw(day) = 6 + day / 7  (assumes speed = 1)
├── Transcriptomic    → estimate_pcw(profile): PCW of the reference with the highest correlation
└── Compare           → error of the two estimates vs the true PCW
```

**To do:** write the functions, simulate 40 organoids (days 30–120, speed uniform in 0.5–1.0) and compare
the errors of the two approaches.

*Discussion question:* the transcriptomic estimate is better, but it uses the same genes that you might
then want to compare between organoid and fetus. What is circular about this? And what if the organoid is
not "a slower fetus" but follows a different trajectory altogether?

In [ ]:
MODULES = {
    "progenitors":   lambda w: 1 / (1 + np.exp((w - 14) / 2)),
    "intermediate":  lambda w: np.exp(-(w - 12) ** 2 / (2 * 2.5 ** 2)),
    "deep_layer":    lambda w: np.exp(-(w - 16) ** 2 / (2 * 3 ** 2)),
    "upper_layer":   lambda w: 1 / (1 + np.exp(-(w - 19) / 1.5)),
    "glia":          lambda w: 1 / (1 + np.exp(-(w - 22) / 1.5)),
}
GENES_PER_MODULE = 10
gene_weights = rng.uniform(0.5, 1.5, size=(len(MODULES), GENES_PER_MODULE))
REFERENCE_PCW = np.arange(8, 24.5, 0.5)

def reference_profile(pcw):
    # TODO: module levels × gene weights → vector of 50 genes
    pass

def simulate_organoid(day, speed, noise=0.8, rng=rng):
    # TODO: return (profile, true_pcw)
    pass

def calendar_pcw(day):
    # TODO
    pass

def estimate_pcw(profile):
    # TODO: correlation with every reference PCW, take the best
    pass

def compare_alignments(n=40):
    # TODO
    pass

## Exercise 3 — Pseudotime from a snapshot (simulation)

**Question:** trajectory methods (Nico's Lecture 3) order cells collected *at the same moment* along a
continuous path. When does this work, and when does it invent an order that is not there?

We simulate cells along a curved differentiation path (almost a full circle in a 2D latent space, so that
the two ends are not far apart in expression space, embedded in 30 genes with noise). The true time `t` goes from 0 (progenitor) to 1 (neuron).

```
Pseudotime from a snapshot
├── Simulate        → simulate_trajectory(n, gap=None): cells at t ~ U(0,1), optionally without t in gap;
│                      latent position (cos 1.8πt, sin 1.8πt), times a random 2×30 matrix, plus noise
├── Method 1        → pc1_pseudotime(X): position along the first principal component
├── Method 2        → graph_pseudotime(X, root, k): shortest-path distance on the k-nearest-neighbour graph
└── Evaluate        → Spearman correlation between pseudotime and true t (only for reachable cells)
```

**To do:** compare the two methods on (i) a dense snapshot and (ii) a snapshot where the intermediate
states (`0.45 < t < 0.6`) were not sampled — for example because they are rare, or because they exist only
for a few days between two harvests.

*Discussion question:* with the gap, the graph is either disconnected (some cells are unreachable) or, if
you increase `k`, it connects through a shortcut. Which of the two failures is more dangerous, and why?
Who decides which end of the trajectory is the "root"?

In [ ]:
N_GENES_TRAJ = 30
embedding = rng.normal(size=(2, N_GENES_TRAJ))

def simulate_trajectory(n=600, gap=None, noise=0.25, rng=rng):
    # TODO: t ~ U(0,1) (excluding the gap), angle = 1.8*pi*t, latent (cos, sin) @ embedding + noise
    pass

def pc1_pseudotime(X):
    # TODO
    pass

def graph_pseudotime(X, root, k=10):
    # TODO: kNN graph with cKDTree, then csgraph.dijkstra from the root
    pass

def evaluate(pseudotime, t):
    # TODO
    pass

## Part B — Sampling in space

## Exercise 4 — A slice of an organoid (simulation)

**Question:** an organoid is a 3D object; a section for histology or spatial transcriptomics is a 2D
slice of it. If an organoid of radius 1 mm contains 10 ventricular-like rosettes (radius 100 µm),
how many rosettes do we see in one section? How often do we see none?

```
Rosettes in one section
├── Organoid        → make_organoid(n_rosettes): random rosette centres inside the sphere (radius 800 µm)
├── Section         → rosettes_in_section(centres, z, r=100): rosettes with |z_centre − z| < r
├── Many sections   → section_survey(n_organoids, sections): counts for random organoids and section heights
└── Summary         → distribution of counts, P(0 rosettes), central vs peripheral sections
```

**To do:** write the functions and compare sections taken at random heights with sections taken
near the centre (|z| < 200 µm), which is what you usually try to do.

*Discussion question:* "organoids under condition X have fewer rosettes" — measured on one section per
organoid. What else could explain the result? (think of organoid size and of where you cut.)
What does stereology recommend?

In [ ]:
ORGANOID_RADIUS = 1000.0
ROSETTE_RADIUS = 100.0

def make_organoid(n_rosettes=10, max_radius=800.0, rng=rng):
    # TODO: rejection sampling of points inside a sphere
    pass

def rosettes_in_section(centres, z, r=ROSETTE_RADIUS):
    # TODO
    pass

def section_survey(n_organoids=2000, central=False, rng=rng):
    # TODO
    pass

## Exercise 5 — Neighbouring bins are not independent

**Question:** a Visium HD section of the mouse brain at 16 µm gives ~100,000 bins. Can we treat them as 100,000 independent
observations in a statistical test?

### 5a — Simulation: two halves of the *same* tissue

We simulate a smooth random field (spatially autocorrelated noise, **no difference** between left and right)
and test "left vs right" with a t-test, first using every bin as an observation, then using blocks of
bins as the unit.

```
False positives with spatial data
├── Field          → smooth_field(shape, smoothness): white noise + Gaussian filter, standardised
├── Bin-level test → t-test left vs right using every bin
├── Block test     → average blocks of 20×20 bins, t-test on block means
└── Repeat         → fraction of p < 0.05 over 200 fields, for both tests
```

*The level of the null hypothesis is always 5%: anything far above that is a broken test.*

In [ ]:
def smooth_field(shape=(100, 100), smoothness=5.0, rng=rng):
    # TODO
    pass

def bin_level_test(field):
    # TODO
    pass

def block_means(field, block=20):
    # TODO: hint, reshape to (h/block, block, w/block, block) and average axes 1 and 3
    pass

def block_level_test(field, block=20):
    # TODO
    pass

def false_positive_rates(n_fields=200, smoothness=5.0):
    # TODO
    pass

### 5b — Real data: how autocorrelated is a Visium HD gene?

`Datasets/visiumhd_mouse_brain_cortex_2um.npz` is the crop of an adult mouse brain Visium HD section used in
Lecture 2 (Exercise 4): 2 µm bins, 67 marker genes. We aggregate it to 16 µm bins.

**Moran's I** measures spatial autocorrelation: ≈ 0 for spatially random values, close to 1 when neighbours
are very similar.

$$I = \frac{N}{W} \frac{\sum_i \sum_j w_{ij}(x_i-\bar x)(x_j-\bar x)}{\sum_i (x_i-\bar x)^2}$$

```
Moran's I of each gene
├── Load         → counts (bins × genes), bins (row, col)
├── Normalise    → log1p(counts / total × median total)
├── Neighbours   → rook neighbours on the grid (up, down, left, right) as a sparse matrix W
├── Moran's I    → morans_i(x, W)
└── Control      → the same gene with bins shuffled in space
```

*Discussion question:* if a gene has Moran's I = 0.6, roughly how many "independent" observations do
100,000 bins correspond to? Is the answer the same for every gene?

In [ ]:
def load_visium(path, size_um=16):
    # load the 2 µm crop and sum it into size_um × size_um bins (same as Lecture 2, Exercise 4)
    z = np.load(path)
    counts = sparse.csr_matrix((z["data"], z["indices"], z["indptr"]), shape=tuple(z["shape"]))
    b = size_um // 2
    key = (z["row"].astype(int) // b) * 100_000 + z["col"].astype(int) // b
    unique, inverse = np.unique(key, return_inverse=True)
    assign = sparse.csr_matrix((np.ones(len(key)), (inverse, np.arange(len(key)))), shape=(len(unique), len(key)))
    bins = pd.DataFrame({"array_row": unique // 100_000, "array_col": unique % 100_000,
                         "total_counts": np.bincount(inverse, weights=z["total_counts"])})
    return (assign @ counts).tocsr(), bins, list(z["genes"])

counts, bins, genes = load_visium("../Datasets/visiumhd_mouse_brain_cortex_2um.npz")
print("16 µm bins:", counts.shape)

def normalise(counts, totals):
    # TODO
    pass

def rook_neighbours(rows, cols):
    # TODO: sparse N×N matrix with 1 where two bins are adjacent on the grid
    pass

def morans_i(x, W):
    # TODO
    pass

## Part C — Sampling populations

## Exercise 6 — How many cells do I need? (analytic + simulation)

**Question:** a population is present at frequency *f* (e.g. outer radial glia at 1%, Cajal–Retzius cells
at 0.2%). How many cells must I sequence to see at least *k* of them with probability *p*?

For *k* = 1 there is a closed formula:
$$P(\text{at least one}) = 1-(1-f)^n \ge p \;\Rightarrow\; n \ge \frac{\ln(1-p)}{\ln(1-f)}$$
To be recognised as a cluster, though, a population needs tens of cells, not one.

```
Cells needed
├── Closed formula     → n_needed_one(f, p)
├── General k          → n_needed(f, p, k): smallest n with P(Binomial(n, f) ≥ k) ≥ p
├── Table              → f ∈ {5%, 1%, 0.5%, 0.1%} × k ∈ {1, 10, 30}, p = 0.95
└── Check by simulation→ draw n cells many times, fraction of draws with ≥ k cells of the population
```

*Discussion question:* the table assumes every cell is captured with the same probability. In Exercise 7d
we will see that this is false. How does that change the numbers? And if the population exists only in 2
of 6 organoids?

In [ ]:
def n_needed_one(f, p=0.95):
    # TODO
    pass

def n_needed(f, p=0.95, k=1):
    # TODO: hint, stats.binom.sf(k - 1, n, f) is P(X >= k)
    pass

def check_by_simulation(f, n, k, n_draws=5000, rng=rng):
    # TODO
    pass

## Exercise 7 — Population traps

Four mini-cases, in the spirit of Exercise 6 of Lecture 1.

| Part | Trap |
|---|---|
| 7a | compositional data: one population grows, all the others seem to shrink |
| 7b | cells vs organoids: the unit of replication for proportions |
| 7c | batch confounded with condition |
| 7d | dissociation bias: the proportions in the dataset are not those in the tissue |

### 7a — One population grows, all the others "shrink"

Four cell types; in condition B **only** type A doubles in absolute number, the other three are unchanged.
6 organoids per condition, 3000 cells sequenced per organoid (always the same number: the machine does not
know how big the organoid was).

```
Compositional effect
├── Simulate      → absolute cell numbers per organoid (log-normal noise), then sample 3000 cells
├── Proportions   → per organoid
├── Test          → t-test of each type's proportion, A vs B
└── Ratio         → test log(type / reference type D) instead, knowing that D did not change
```

*Discussion question:* without external information (absolute counts, spike-in cells, a population
known not to change) can you tell "A increased" from "B, C, D decreased"?

In [ ]:
CELL_TYPES = ["A", "B", "C", "D"]
BASE_ABUNDANCE = np.array([1000, 3000, 4000, 2000])

def simulate_composition(fold_a=1.0, n_organoids=6, cells_sequenced=3000, rng=rng):
    # TODO: absolute numbers with log-normal noise, then rng.multinomial(cells_sequenced, ...)
    pass

def proportions(counts):
    # TODO
    pass

def compare_conditions(control, treated):
    # TODO
    pass

### 7b — Cells or organoids?

The fraction of a cell type varies between organoids (organoid-to-organoid variability), but there is
**no** difference between conditions. We compare two tests, repeated over 500 simulated experiments:

```
Unit of replication for proportions
├── Simulate      → 4 organoids per condition, proportion ~ Beta(mean 20%, between-organoid sd ≈ 5%), 2000 cells each
├── Cells as n    → chi-square test on the 2×2 table of pooled cell counts
├── Organoids as n→ t-test on the 4 + 4 per-organoid proportions
└── Repeat        → false positive rate of each test
```

*Discussion question:* the pooled test has 16,000 "observations". How many does it really have?

In [ ]:
def simulate_fractions(n_organoids=4, cells=2000, mean=0.2, concentration=60, rng=rng):
    # TODO: per-organoid proportion from a Beta, then binomial cell counts
    pass

def pooled_cell_test(k1, n1, k2, n2):
    # TODO
    pass

def organoid_test(k1, n1, k2, n2):
    # TODO
    pass

def false_positives(n_experiments=500):
    # TODO
    pass

### 7c — Batch confounded with condition

All control organoids were differentiated in January, all treated ones in March. The treatment does
**nothing**; the March batch has a +1 shift in the measured quantity.

```
Confounded vs balanced design
├── Simulate      → simulate_design(design): 8 organoids, batch effect +1, treatment effect 0
├── Confounded    → control = batch 1, treated = batch 2
├── Balanced      → each batch has 2 controls and 2 treated
└── Estimate      → least squares with columns [intercept, treated, batch] (when possible)
```

*Discussion question:* in the confounded design, why does `np.linalg.lstsq` not give you a separate
treatment effect? Can any analysis rescue that experiment?

In [ ]:
def simulate_design(design, batch_effect=1.0, treatment_effect=0.0, rng=rng):
    # TODO
    pass

def naive_difference(df):
    # TODO
    pass

def adjusted_estimate(df):
    # TODO: least squares; return NaN if the design matrix is rank deficient
    pass

### 7d — Where did the neurons go?

Dissociation kills or loses fragile cells (mature neurons with long processes) more than compact ones
(progenitors). Suppose the true composition of the tissue and the probability that a cell of each type
survives dissociation and capture are:

| type | tissue | capture |
|---|---|---|
| progenitors | 30% | 0.8 |
| intermediate progenitors | 10% | 0.7 |
| neurons | 50% | 0.3 |
| glia | 10% | 0.6 |

```
Dissociation bias
├── Observed     → expected_observed(tissue, capture): tissue × capture, renormalised
├── Simulate     → sample 5000 cells from the tissue, keep each with its capture probability
└── Correct      → divide observed by capture and renormalise (needs capture to be known!)
```

*Discussion question:* how would you estimate the capture probabilities in practice? (hint: spatial
transcriptomics or imaging on the same tissue, single-nucleus vs single-cell.)

In [ ]:
tissue = pd.Series({"progenitors": 0.30, "intermediate": 0.10, "neurons": 0.50, "glia": 0.10})
capture = pd.Series({"progenitors": 0.8, "intermediate": 0.7, "neurons": 0.3, "glia": 0.6})

def expected_observed(tissue, capture):
    # TODO
    pass

def simulate_capture(tissue, capture, n_cells=5000, rng=rng):
    # TODO
    pass

def correct(observed, capture):
    # TODO
    pass

## Wrap-up

For each level (time, space, populations), write in one line: what can go wrong, and what you would do
in the design of an organoid experiment to prevent it.